# Data Wrangling using Pandas

## Dataset Description: Student Assessment Questionnaires

The dataset `assessment_generated.csv` contains information derived from student assessment questionnaires.

Each record represents an individual student's response and includes demographic, academic, and self-assessment information. The dataset comprises the following attributes:

- **`timestamp`**  
  The date and time when the assessment was submitted, formatted as `yyyy-mm-dd hh:mm:ss timezone`.

- **`netid`**  
  The encoded NetID of the student. Valid NetIDs must have a string length between 8 and 14 characters (inclusive). Entries falling outside this range are considered invalid.

- **`ruid`**  
  The encoded RUID of the student. A valid RUID is expected to contain exactly 18 characters. Any deviation from this length is considered invalid.

- **`section`**  
  The course section number as reported by the student. This field may contain inaccuracies, as some students provided incorrect section information.

- **`role`**  
  The academic standing of the student. Possible values include:
  - `Freshman`
  - `Sophomore`
  - `Junior`
  - `Senior`
  - `Graduate`
  - `Other`

- **`major`**  
  The declared major of the student. Accepted categories are:
  - `Computer Science`
  - `Electrical and Computer Engineering`
  - `Mathematics`
  - `Other`

- **Skill Proficiency Columns**  
  The following columns record students’ self-assessed proficiency levels in specific skills, rated on scales ranging from 0 up to a multiple of 5 (depending on the number of questions per topic). Missing values are present in some entries.

  - `data_structures`  
  - `calculus_and_linear_algebra`  
  - `probability_and_statistics`  
  - `data_visualization`  
  - `python_libraries`  
  - `shell_scripting`  
  - `sql`  
  - `python_scripting`  
  - `jupyter_notebook`  
  - `regression`  
  - `programming_languages`  
  - `algorithms`  
  - `complexity_measures`  
  - `visualization_tools`  
  - `massive_data_processing`


## Tasks

- Import Data
  - Load assessment_generated.xlsx as a DataFrame named student_assessment_xlsx.
- Verify NetIDs
  - Ensure that student_assessment_xlsx contains the same set of netids as student_assessment.
- Analyze RUID Lengths
  - Display the frequency of each ruid length.
  - Display records where ruid length exceeds 20 characters.
- Compute Total Score
  - Create a new column total_score as the sum of all skill proficiency columns.
  - Sort records by total_score in descending order.
- Group Statistics by Section
  - Group students by section and compute the mean, median, and standard deviation for each skill proficiency column.
- Pivot Table by Role and Section
  - Create a pivot table where:
    - Rows correspond to role
    - Columns correspond to section numbers
    - Entries contain the average total_score
- Format Timestamp
  - Convert timestamp values to the EST timezone instead of UTC.
- Normalize Skill Proficiency
  - For each proficiency column, apply z-score normalization rather than min-max scaling.
- Handle Missing Values
  - Fill missing values in each skill proficiency column with the column mean.
- Remove Duplicate Records
  - Drop duplicates while keeping only the record with the highest total_score.
- Resolve Swapped IDs
  - Identify records where students may have swapped netid and ruid. (Hint: netid should be shorter than ruid.)
- For records swapped netid and ruid, attempt a join using:
  - student_assessment.ruid = student_list.netid


### Setup Code (Please run this first to set up the environment)

In [1]:
import numpy as np
import pandas as pd
from sklearn import preprocessing

In [2]:
if __name__ == "__main__":
    skill_columns = [
        'data_structures', 'calculus_and_linear_algebra', 'probability_and_statistics',
        'data_visualization', 'python_libraries', 'shell_scripting', 'sql',
        'python_scripting', 'jupyter_notebook', 'regression', 'programming_languages',
        'algorithms', 'complexity_measures', 'visualization_tools', 'massive_data_processing'
    ]

    CSV_PATH = 'assessment_generated.csv'
    # load csv file to a Pandas dataframe named student_assessments
    student_assessments_csv = pd.read_csv(CSV_PATH)
    display(student_assessments_csv)

    STUDENT_LIST_PATH = 'student_list_generated.csv'
    student_list_df = pd.read_csv(STUDENT_LIST_PATH)

,timestamp,netid,ruid,section,role,major,data_structures,calculus_and_linear_algebra,probability_and_statistics,data_visualization,...,shell_scripting,sql,python_scripting,jupyter_notebook,regression,programming_languages,algorithms,complexity_measures,visualization_tools,massive_data_processing
0,2026-09-04 01:24:03 +0300,d2dbd3d0d5,786a2021217c6e2022,1,Junior,Computer Science,28.0,20.0,38.0,27.0,...,5.0,5.0,0.0,5.0,13.0,21.0,20.0,3.0,14.0,9.0
1,2026-09-04 00:30:39 +0200,c7dd9ac7c494,60703e393965793e3d,1,Junior,Computer Science,8.0,12.0,13.0,12.0,...,0.0,4.0,1.0,NaN,0.0,14.0,9.0,5.0,0.0,2.0
2,2026-09-03 18:19:47 -0400,5d504543461b,0f1d55565609195250,1,Senior,Mathematics,23.0,16.0,22.0,23.0,...,NaN,5.0,0.0,0.0,3.0,14.0,NaN,NaN,8.0,2.0
3,2026-09-04 06:26:53 +0800,021b4e0503,5145080b0b52450c0a,3,Senior,Computer Science,30.0,19.0,58.0,28.0,...,1.0,5.0,5.0,1.0,NaN,26.0,31.0,10.0,NaN,8.0
4,2026-09-03 16:21:34 -0600,8b8cc28089de,5d4c0104045b490005,1,Junior,Computer Science,26.0,13.0,44.0,22.0,...,1.0,5.0,1.0,1.0,13.0,13.0,7.0,0.0,7.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,2026-09-04 00:27:28 +0200,a89a979d95,8b9bdad2d28e92d2da,2,Sophomore,Mathematics,18.0,NaN,27.0,16.0,...,1.0,5.0,0.0,0.0,3.0,18.0,11.0,3.0,19.0,4.0
153,2026-09-03 22:30:08 +0000,eaecb4edea,8d9dd4d4d48f99d0d0,2,Senior,Computer Science,15.0,6.0,8.0,14.0,...,1.0,5.0,1.0,1.0,0.0,7.0,8.0,0.0,1.0,0.0
154,2026-09-03 18:29:26 -0400,e7d58dcecf9e,bbaae7e2e2b1aceae7,1,Senior,Computer Science,13.0,9.0,11.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
155,2026-09-03 17:16:09 -0500,eef7a3e5e6,03125d5a5a06105b5d,3,Senior,Electrical and Computer Engineering,35.0,18.0,42.0,33.0,...,1.0,0.0,0.0,5.0,5.0,25.0,25.0,5.0,13.0,0.0


### Import Data

In [17]:
def import_data(path):
    """
    Load assessment_generated.xlsx as a DataFrame 
    IN: path, str, path to the file
    OUT: student_assessment_xlsx, pd.DataFrame
    """
    # Your_Code_Here
    student_assessment_xlsx = pd.read_excel(path)
    # print(student_assessment_xlsx.head())

    return student_assessment_xlsx

In [18]:
if __name__ == "__main__":
    XLSX_PATH = 'assessment_generated.xlsx'
    # load xlsx file to a Pandas dataframe named student_assessments
    student_assessments_xlsx = import_data(XLSX_PATH)

### Verify NetIDs

In [ ]:
def verify_netids(df_csv, df_xlsx):
    """
    Verify that the NetIDs in the CSV and XLSX files match
    IN: df_csv, pd.DataFrame, dataframe from CSV file
        df_xlsx, pd.DataFrame, dataframe from XLSX file
    OUT: flag, bool, True if NetIDs match, False otherwise
    """
    # Your_Code_Here
    flag = df_csv['netid'].equals(df_xlsx['netid']) # netid column is all lowercase

    return flag

In [28]:
if __name__ == "__main__":
    netid_same_flag = verify_netids(student_assessments_csv, student_assessments_xlsx)
    print(f"NetIDs match: {netid_same_flag}")

NetIDs match: True


### Analyze RUID Lengths

In [29]:
def get_ruid_length_freq(df):
    """
    Get the frequency of RUID lengths in the dataframe
    IN: df, pd.DataFrame, dataframe containing RUIDs
    OUT: ruid_length_freq, dict, dictionary with RUID lengths as keys and their frequencies as values
    """
    # Your_Code_Here
    lengths = df['ruid'].astype(str).str.len() # Transform ruid column to string and get length of each ruid
    ruid_length_freq = lengths.groupby(lengths).size().to_dict() # Group by length and count frequency then convert into dictionary
    return ruid_length_freq

In [30]:
if __name__ == "__main__":
    ruid_length_freq = get_ruid_length_freq(student_assessments_xlsx)
    for length, freq in sorted(ruid_length_freq.items()):
        print(f"RUID Length: {length:<2}, Frequency: {freq:<3}")

RUID Length: 4 , Frequency: 1  
RUID Length: 6 , Frequency: 1  
RUID Length: 10, Frequency: 2  
RUID Length: 16, Frequency: 2  
RUID Length: 18, Frequency: 147
RUID Length: 20, Frequency: 3  
RUID Length: 28, Frequency: 1  


In [ ]:
def get_ruid_length_outliers(df):
    """
    Identify RUIDs with lengths that are > 20 characters
    IN: df, pd.DataFrame, dataframe containing RUIDs
    OUT: outliers, pd.DataFrame, dataframe containing outlier RUIDs
    """
    # Your_Code_Here
    outliers = df[df['ruid'].astype(str).str.len() > 20] # Filter a DF containing RUIDs with length > 20
    # There was only 1 outlier from the previous result

    return outliers


In [34]:
if __name__ == "__main__":
    outliers = get_ruid_length_outliers(student_assessments_xlsx)
    display(outliers)

,timestamp,netid,ruid,section,role,major,data_structures,calculus_and_linear_algebra,probability_and_statistics,data_visualization,...,shell_scripting,sql,python_scripting,jupyter_notebook,regression,programming_languages,algorithms,complexity_measures,visualization_tools,massive_data_processing
33,2026-09-04 06:44:55 +0800,534b5f535e,e3fabab9bae1f7b2bcbee8f1beb8,4,Junior,Computer Science,13.0,14.0,42.0,29.0,...,1.0,NaN,1.0,2.0,0.0,NaN,2.0,0.0,5.0,0.0


### Compute Total Score

Note: Skip `nan` entries when computing sum.

In [36]:
def compute_total_score(df, skill_columns):
    """
    Compute the total score for each student
    IN: df, pd.DataFrame, dataframe containing student assessments
    IN: skill_columns, list of str, list of columns to sum for total score
    OUT: df, pd.DataFrame, dataframe with total_score column
    """
    # Your_Code_Here
    df['total_score'] = df[skill_columns].sum(axis=1) # Axis = 0 goes down entries, Axis = 1 goes across features
    
    return df

In [ ]:
def sort_by_total_score(df):
    """
    Sort the dataframe by total score in descending order
    IN: df, pd.DataFrame, dataframe containing total_score column
    OUT: df_sorted, pd.DataFrame, sorted dataframe
    """
    # Your_Code_Here
    df_sorted = df.sort_values(by='total_score', ascending=False) # Order the dataframe by descending total score column

    return df_sorted

In [38]:
if __name__ == "__main__":
    student_assessments_with_total_score = compute_total_score(student_assessments_xlsx.copy(), skill_columns)
    student_assessments_with_total_score = sort_by_total_score(student_assessments_with_total_score)
    display(student_assessments_with_total_score)

,timestamp,netid,ruid,section,role,major,data_structures,calculus_and_linear_algebra,probability_and_statistics,data_visualization,...,sql,python_scripting,jupyter_notebook,regression,programming_languages,algorithms,complexity_measures,visualization_tools,massive_data_processing,total_score
10,2026-09-03 12:39:05 -1000,54490a53500a,fceeaca5a5fbeea3ac,3,Senior,Computer Science,35.0,14.0,45.0,35.0,...,5.0,5.0,4.0,15.0,21.0,26.0,10.0,13.0,6.0,254.0
14,2026-09-03 13:34:53 -0900,f3edfaf9faa5,b8a9e2e1e1baaee7e4,1,Sophomore,Computer Science,25.0,19.0,46.0,35.0,...,5.0,5.0,5.0,9.0,24.0,22.0,9.0,21.0,8.0,253.0
60,2026-09-04 01:22:32 +0300,1b0f13121349,2d3f7274742f387074,1,Senior,Computer Science,34.0,25.0,39.0,35.0,...,1.0,5.0,5.0,12.0,27.0,28.0,5.0,NaN,12.0,248.0
20,2026-09-04 04:22:41 +0600,f9e1fefbff,7766292e2e7161292d,1,Junior,Computer Science,35.0,16.0,51.0,35.0,...,0.0,5.0,5.0,12.0,22.0,22.0,6.0,14.0,5.0,248.0
50,2026-09-03 22:17:39 +0000,6668646765,05175f5c5c03145d5c,3,Junior,Computer Science,25.0,17.0,53.0,28.0,...,5.0,5.0,5.0,NaN,28.0,26.0,8.0,23.0,NaN,243.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,2026-09-03 18:29:26 -0400,e7d58dcecf9e,bbaae7e2e2b1aceae7,1,Senior,Computer Science,13.0,9.0,11.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.0
95,2026-09-04 01:38:27 +0300,87,6071,1,Junior,Computer Science,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,NaN,4.0
113,2026-09-03 12:22:09 -1000,9491999b9dc3,ccde919595cbdb9095,1,Junior,Computer Science,1.0,NaN,0.0,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,3.0
125,2026-09-04 01:18:30 +0300,6a676c6d6c,0415585d5d0317555b,1,Senior,Computer Science,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,1.0,3.0


### Group Statistics by Section

In [ ]:
def get_section_statistics(df, skill_columns):
    """
    Group students by section and compute the mean, median, and standard deviation for each skill proficiency column
    IN: df, pd.DataFrame, dataframe containing student assessments
    IN: skill_columns, list of str, list of skill proficiency columns
    OUT: section_stats, pd.DataFrame, dataframe with section statistics
    """
    # Your_Code_Here
    return section_stats

In [ ]:
if __name__ == "__main__":
    section_statistics = get_section_statistics(student_assessments_with_total_score.copy(), skill_columns)
    display(section_statistics)

### Pivot Table by Role and Section

In [ ]:
def create_pivot_table(df):
    """
    Create a pivot table where rows correspond to role, columns correspond to section numbers, and entries contain the average total_score
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: pivot_table, pd.DataFrame, pivot table
    """
    # set role as categorical with specified order
    # Your_Code_Here

    # generate pivot table
    # Your_Code_Here
    
    return pivot_table

In [ ]:
if __name__ == "__main__":
    pivot_table = create_pivot_table(student_assessments_with_total_score.copy())
    display(pivot_table)

### Format Timestamp

In [ ]:
def format_timestamp(df):
    """
    Convert timestamp values to the EST timezone instead of UTC
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: df, pd.DataFrame, dataframe with formatted timestamp
    """
    # Your_Code_Here
    return df

In [ ]:
if __name__ == "__main__":
    student_assessments_formatted_timestamp = format_timestamp(student_assessments_with_total_score.copy())
    display(student_assessments_formatted_timestamp)

### Normalize Skill Proficiency

In [ ]:
def normalize_skills(df, skill_columns):
    """
    For each proficiency column, apply z-score normalization
    IN: df, pd.DataFrame, dataframe containing student assessments
    IN: skill_columns, list of str, list of skill proficiency columns
    OUT: df, pd.DataFrame, dataframe with normalized skills
    """
    # Your_Code_Here
    return df

In [ ]:
if __name__ == "__main__":
    student_assessments_normalized = normalize_skills(student_assessments_formatted_timestamp.copy(), skill_columns)
    display(student_assessments_normalized)

### Handle Missing Values

In [ ]:
def handle_missing_values(df, skill_columns):
    """
    Fill missing values in each skill proficiency column with the column mean
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: df, pd.DataFrame, dataframe with missing values handled
    """
    # Your_Code_Here
    return df

In [ ]:
if __name__ == "__main__":
    student_assessments_no_missing = handle_missing_values(student_assessments_normalized.copy(), skill_columns)
    display(student_assessments_no_missing)

### Remove Duplicate Records

Note: Drop duplicates based on `ruid`. Keep the record with the highest `total_score`. If there is a tie, keep the one whose `timestamp` is latest.

In [ ]:
def remove_duplicates(df):
    """
    Drop duplicates while keeping only the record with the highest total_score
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: df, pd.DataFrame, dataframe with duplicates removed
    """
    # sort by total_score descending, then by timestamp descending
    # Your_Code_Here

    # drop duplicates
    # Your_Code_Here
    
    # sort back
    # Your_Code_Here
    return df

In [ ]:
if __name__ == "__main__":
    student_assessments_no_duplicates = remove_duplicates(student_assessments_no_missing.copy())
    display(student_assessments_no_duplicates)

### Resolve Swapped IDs

In [ ]:
def get_swapped_records(df):
    """
    Identify records where students may have swapped netid and ruid
    IN: df, pd.DataFrame, dataframe containing student assessments
    OUT: swapped_df, pd.DataFrame, dataframe containing swapped records
    """
    # calculate lengths
    # Your_Code_Here

    # find netid with exactly 18 characters and ruid length between 8 and 14
    # Your_Code_Here
    
    return swapped_df

In [ ]:
if __name__ == "__main__":
    swapped_records = get_swapped_records(student_assessments_no_duplicates.copy())
    display(swapped_records)

In [ ]:
def join_swapped_records(df_swapped, student_list_df):
    """
    Join the swapped records with the student list to correct netid and ruid
    IN: df_swapped, pd.DataFrame, dataframe containing swapped records
    IN: student_list_df, pd.DataFrame, dataframe containing student list
    OUT: joined_df, pd.DataFrame, dataframe with corrected netid and ruid
    """
    # Your_Code_Here
    return joined_df

In [ ]:
if __name__ == "__main__":
    joined_swapped = join_swapped_records(swapped_records.copy(), student_list_df)
    display(joined_swapped)

### DataFrame Schema

In [ ]:
if __name__ == "__main__":
    section_statistics.info()

In [ ]:
if __name__ == "__main__":
    pivot_table.info()

In [ ]:
if __name__ == "__main__":
    student_assessments_no_duplicates.info()

In [ ]:
if __name__ == "__main__":
    joined_swapped.info()